# GovPrep-AI — Phase 11: BERT Fine-Tuning (Google Colab)

This notebook fine-tunes the **existing GovPrep-AI BERT implementation** for the four-class `subject` task.

It reuses the exact processed files already used by TF-IDF, LSTM and BiLSTM:

- `data/processed/public_train.csv`
- `data/processed/public_val.csv`
- `data/processed/public_test.csv`

**Do not run `training.prepare_data` in this notebook.** Re-splitting would make the BERT comparison invalid.

## Before you start

1. In Colab, select **Runtime → Change runtime type → T4 GPU** (or another available GPU).
2. Enable internet access so Hugging Face can download `bert-base-uncased`.
3. On your Windows machine, create the small source/data bundle with the PowerShell command shown in the next section.
4. Upload that ZIP when prompted.

The notebook saves BERT artifacts in `nlp_service/artifacts/bert/` and then packages them for download back to your project.

**Data limitation:** the current dataset supports subject classification only. Topic and difficulty labels are not available, so this notebook trains only `subject` and does not fabricate those targets.


## 1. Create the upload bundle on Windows

Open PowerShell in `D:\Projects\gp2\nlp_service` and run this command. It includes source code and the already-prepared splits, but excludes `.venv311`, raw downloads and existing model artifacts.

```powershell
cd D:\Projects\gp2\nlp_service

$stage = Join-Path $env:TEMP "govprep_bert_bundle"
$zip = Join-Path $PWD "govprep_bert_bundle.zip"

if (Test-Path $stage) { Remove-Item $stage -Recurse -Force }
if (Test-Path $zip) { Remove-Item $zip -Force }

New-Item -ItemType Directory -Force "$stage\data\processed" | Out-Null
Copy-Item .\app "$stage\app" -Recurse
Copy-Item .\training "$stage\training" -Recurse
Copy-Item .\data\processed\public_train.csv "$stage\data\processed\public_train.csv"
Copy-Item .\data\processed\public_val.csv "$stage\data\processed\public_val.csv"
Copy-Item .\data\processed\public_test.csv "$stage\data\processed\public_test.csv"
if (Test-Path .\data\processed\split_manifest.json) {
    Copy-Item .\data\processed\split_manifest.json "$stage\data\processed\split_manifest.json"
}

Compress-Archive -Path "$stage\*" -DestinationPath $zip
Write-Host "Created: $zip"
```

This does not modify or delete your project files. Upload the resulting `govprep_bert_bundle.zip` in the next cell.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

uploaded = files.upload()  # Choose govprep_bert_bundle.zip from your Windows machine
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise RuntimeError(f"Please upload exactly one ZIP bundle; received: {list(uploaded)}")

work_root = Path('/content/govprep_ai_bert_work')
if work_root.exists():
    shutil.rmtree(work_root)
work_root.mkdir(parents=True)

bundle_path = Path('/content') / zip_names[0]
with zipfile.ZipFile(bundle_path) as zf:
    # Basic ZIP path-traversal guard
    for member in zf.infolist():
        target = (work_root / member.filename).resolve()
        if not str(target).startswith(str(work_root.resolve()) + os.sep) and target != work_root.resolve():
            raise RuntimeError(f"Unsafe path in ZIP: {member.filename}")
    zf.extractall(work_root)

nlp_root = work_root
required = [
    nlp_root / 'app' / 'bert.py',
    nlp_root / 'training' / 'train_bert.py',
    nlp_root / 'data' / 'processed' / 'public_train.csv',
    nlp_root / 'data' / 'processed' / 'public_val.csv',
    nlp_root / 'data' / 'processed' / 'public_test.csv',
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Bundle is incomplete. Missing:\n' + '\n'.join(missing))

os.chdir(nlp_root)
import sys
if str(nlp_root) not in sys.path:
    sys.path.insert(0, str(nlp_root))

print('Bundle extracted to:', nlp_root)
print('Processed train/validation/test files found.')


## 2. Install BERT dependencies

Colab normally includes PyTorch. This cell installs the Hugging Face Transformers package and the project configuration/data dependencies without reinstalling PyTorch.


In [ ]:
%pip -q install "transformers>=4.40" "pydantic>=2.6" "pydantic-settings>=2.2" "scikit-learn>=1.4" "pandas>=2.1" "nltk>=3.8.1"


In [ ]:
import torch, transformers, pandas as pd, json
from pathlib import Path

print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. BERT fine-tuning may be very slow on CPU.')

for split in ['train', 'val', 'test']:
    path = nlp_root / 'data' / 'processed' / f'public_{split}.csv'
    df = pd.read_csv(path)
    print(f"{split}: {len(df)} rows; columns={list(df.columns)}")
    if 'subject' not in df.columns or 'text' not in df.columns:
        raise ValueError(f'{path.name} must contain text and subject columns')
    print('  subject labels:', df['subject'].fillna('').value_counts().to_dict())

if not torch.cuda.is_available():
    raise RuntimeError('GPU not detected. In Colab select Runtime > Change runtime type > T4 GPU, then rerun this cell.')


## 3. Fine-tune pretrained BERT for subject classification

Configuration is intentionally conservative for a Colab T4 GPU:

- Checkpoint: `bert-base-uncased` (pretrained; not trained from scratch)
- Maximum sequence length: 128
- Batch size: 8
- Maximum epochs: 4
- Learning rate: `2e-5`
- Early stopping: patience 2 on validation Macro F1
- Device: `auto` (uses CUDA when available)

The test split is evaluated only after the best epoch is selected using validation Macro F1.


In [ ]:
import subprocess, sys, os

cmd = [
    sys.executable, '-m', 'training.train_bert',
    '--targets', 'subject',
    '--seed', '42',
    '--pretrained', 'bert-base-uncased',
    '--max-len', '128',
    '--batch-size', '8',
    '--epochs', '4',
    '--lr', '2e-5',
    '--patience', '2',
    '--device', 'auto',
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=str(nlp_root), text=True)
if result.returncode != 0:
    raise RuntimeError(f'BERT training failed with exit code {result.returncode}. Review the error above.')


## 4. Review actual test metrics

This cell reads the `metrics.json` written by the training script. It does not create or hardcode results.


In [ ]:
bert_dir = nlp_root / 'artifacts' / 'bert'
metrics_path = bert_dir / 'metrics.json'
if not metrics_path.is_file():
    raise FileNotFoundError('BERT metrics.json not found; training did not complete successfully.')

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
subject = metrics.get('targets', {}).get('subject', {})
if subject.get('status') != 'trained':
    raise RuntimeError(f"Subject target is not trained: {subject}")

print('Pretrained model:', metrics.get('pretrained', {}).get('name'))
print('Device:', metrics.get('framework', {}).get('device'))
print('Best epoch:', subject.get('training', {}).get('best_epoch'))
print('Training seconds:', subject.get('training', {}).get('train_seconds'))
for split in ['val', 'test']:
    m = subject.get('splits', {}).get(split)
    if m:
        print(f"{split}: n={m['n']} accuracy={m['accuracy']:.4f} macro-F1={m['f1_macro']:.4f} weighted-F1={m['f1_weighted']:.4f}")

print('Artifacts:', [p.name for p in bert_dir.iterdir()])


## 5. Package BERT artifacts for the Windows project

The ZIP will contain a top-level `bert/` folder. Extract it into:

`D:\Projects\gp2\nlp_service\artifacts`

This places the model checkpoint, tokenizer files and metrics under `artifacts/bert/` without touching TF-IDF, Word2Vec, LSTM or BiLSTM artifacts.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

artifact_zip = Path('/content/govprep_bert_artifacts.zip')
with ZipFile(artifact_zip, 'w', compression=ZIP_DEFLATED) as zf:
    for path in bert_dir.rglob('*'):
        if path.is_file():
            zf.write(path, arcname=str(Path('bert') / path.relative_to(bert_dir)))

print('Created:', artifact_zip, 'size (MB):', round(artifact_zip.stat().st_size / (1024 * 1024), 2))
files.download(str(artifact_zip))


## 6. Restore artifacts on Windows

After downloading `govprep_bert_artifacts.zip`, open PowerShell:

```powershell
cd D:\Projects\gp2\nlp_service
Expand-Archive -Path "$env:USERPROFILE\Downloads\govprep_bert_artifacts.zip" -DestinationPath ".\artifacts" -Force

Get-ChildItem .\artifacts\bert -Recurse |
    Select-Object FullName, Length
```

Expected important files:

- `artifacts/bert/subject.pt`
- `artifacts/bert/tokenizer/` (tokenizer configuration and vocabulary files)
- `artifacts/bert/metrics.json`

Keep the downloaded ZIP and `metrics.json` as experimental evidence for Phase 12. Do not report results until the actual Colab training run has completed successfully.

### If CUDA out-of-memory occurs

Restart the Colab runtime and rerun the notebook with `--batch-size 4` and `--max-len 96`. Do not change the dataset split. Record the changed hyperparameters in the final report.
